# Phase 0c: Train Stage 1 Kaggle-Only (100% Kaggle Data)

## Contexte

Le Stage 1 original entraîné sur données mixtes (60% Kaggle + 40% externes) a échoué:
- **Validation**: 80.5%
- **Test (Kaggle-only)**: **15.5%** ❌

**Root Cause**: Distribution shift massif car test set = 100% Kaggle.

## Objectif

Entraîner Stage 1 sur **100% données Kaggle équilibrées** (Phase 0b):
- Train: 16,000 images (4,000/classe × 5 classes × 80%)
- Val: 4,000 images (4,000/classe × 5 classes × 20%)
- **Cible**: ≥70% test accuracy

## Configuration

| Paramètre | Valeur | Raison |
|-----------|--------|--------|
| Epochs | 5 | Plus car dataset plus petit |
| Learning Rate | 2e-5 | Plus agressif |
| LoRA r | 16 | Capacité suffisante |
| Batch (effective) | 16 | 4 × 4 gradient accum |

## 1. Imports et Configuration

In [ ]:
# ============================================================
# IMPORTS
# ============================================================

import os
import json
import re
import random
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from PIL import Image
from dataclasses import dataclass
from typing import Dict, List, Any, Optional
from collections import defaultdict, Counter
from datasets import Dataset
from tqdm import tqdm

# IMPORTANT: Import unsloth AVANT transformers
from unsloth import FastVisionModel

from transformers import (
    AutoProcessor,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# ============================================================
# CONFIGURATION KAGGLE-ONLY STAGE 1
# ============================================================

# Chemins
BASE_DIR = Path("/home/hounfodjidagba/learning/cassava/new_implementation")
DATA_DIR = BASE_DIR / "data" / "kaggle_only"  # Dataset Kaggle-only de Phase 0b
CHECKPOINT_DIR = BASE_DIR / "checkpoints" / "stage1_kaggle_only"  # Nouveau checkpoint
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Modèle
MODEL_NAME = "unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit"
TARGET_IMAGE_SIZE = (512, 512)

# ============================================================
# LoRA Configuration - Augmentée pour mieux apprendre
# ============================================================
LORA_CONFIG = {
    "r": 16,                    # Plus de capacité
    "lora_alpha": 32,           # ratio 1:2
    "lora_dropout": 0.05,       # Petit dropout
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],  # Plus de modules
}

# ============================================================
# Training Configuration - Ajustée pour dataset plus petit
# ============================================================
TRAINING_CONFIG = {
    "per_device_train_batch_size": 4,     # Batch size
    "gradient_accumulation_steps": 4,      # Effective batch = 16
    "num_train_epochs": 5,                 # Plus d'epochs (dataset plus petit)
    "learning_rate": 2e-5,                 # Learning rate standard
    "warmup_ratio": 0.15,                  # Plus de warmup
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,
    "logging_steps": 50,
    "save_steps": 500,
    "eval_steps": 250,
    "bf16": torch.cuda.is_bf16_supported(),
    "fp16": not torch.cuda.is_bf16_supported(),
}

# Classes
CLASS_NAMES = [
    "Cassava Bacterial Blight (CBB)",
    "Cassava Brown Streak Disease (CBSD)",
    "Cassava Green Mottle (CGM)",
    "Cassava Mosaic Disease (CMD)",
    "Healthy"
]

# Seed
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print(f"{'='*60}")
print(f"CONFIGURATION STAGE 1 KAGGLE-ONLY")
print(f"{'='*60}")
print(f"\nModèle: {MODEL_NAME}")
print(f"\nLoRA: r={LORA_CONFIG['r']}, alpha={LORA_CONFIG['lora_alpha']}")
print(f"Modules: {LORA_CONFIG['target_modules']}")
print(f"\nTraining:")
print(f"  - LR: {TRAINING_CONFIG['learning_rate']}")
print(f"  - Epochs: {TRAINING_CONFIG['num_train_epochs']}")
print(f"  - Batch: {TRAINING_CONFIG['per_device_train_batch_size']} x {TRAINING_CONFIG['gradient_accumulation_steps']}")
print(f"{'='*60}")

## 2. Chargement du Modèle

In [ ]:
# ============================================================
# CHARGEMENT MODÈLE QWEN2.5-VL-7B
# ============================================================

print("\n" + "="*60)
print("LOADING QWEN2.5-VL-7B")
print("="*60)

# Charger le modèle
model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

# Ajouter LoRA
model = FastVisionModel.get_peft_model(
    model,
    r=LORA_CONFIG["r"],
    lora_alpha=LORA_CONFIG["lora_alpha"],
    lora_dropout=LORA_CONFIG["lora_dropout"],
    target_modules=LORA_CONFIG["target_modules"],
    use_rslora=False,
    use_gradient_checkpointing="unsloth"
)

# Charger le processor
processor = AutoProcessor.from_pretrained(MODEL_NAME)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"\n✅ Modèle chargé: {MODEL_NAME}")
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.4f}%)")
print("="*60)

## 3. Chargement des Données Kaggle-Only

In [ ]:
# ============================================================
# CHARGEMENT DONNÉES KAGGLE-ONLY (Phase 0b)
# ============================================================

print("\n" + "="*60)
print("LOADING KAGGLE-ONLY DATA")
print("="*60)

def load_json_dataset(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

# Charger datasets Kaggle-only
train_file = DATA_DIR / "cassava_train.json"
val_file = DATA_DIR / "cassava_val.json"

if not train_file.exists() or not val_file.exists():
    print("❌ ERREUR: Datasets Kaggle-only non trouvés!")
    print(f"   Veuillez d'abord exécuter le notebook 16_prepare_kaggle_dataset.ipynb")
    print(f"   pour créer les fichiers:")
    print(f"     - {train_file}")
    print(f"     - {val_file}")
    raise FileNotFoundError("Kaggle-only datasets not found")

cassava_train = load_json_dataset(train_file)
cassava_val = load_json_dataset(val_file)

print(f"\n✅ Données chargées:")
print(f"  Train: {len(cassava_train):,}")
print(f"  Val:   {len(cassava_val):,}")

# Charger stats
stats_file = DATA_DIR / "dataset_stats.json"
if stats_file.exists():
    with open(stats_file, 'r') as f:
        stats = json.load(f)
    print(f"\n📊 Distribution (équilibrée):")
    for cls, count in stats.get('distribution_after', {}).items():
        print(f"  {cls}: {count}")

## 4. Formatage pour Qwen

In [ ]:
# ============================================================
# PROMPT FORMAT - Utilise le même format que l'ancien Stage 1
# Compatible avec l'évaluation existante
# ============================================================

# Format simple pour maximiser la discrimination
QUESTION = "What disease is affecting this cassava plant?"

def extract_image_path(text):
    """Extrait le chemin de l'image du texte."""
    match = re.search(r'<img>(.*?)</img>', text)
    return match.group(1) if match else None

def format_sample_for_qwen(sample):
    """
    Formate un sample Kaggle-only pour l'entraînement Qwen.
    
    Format d'entrée (de Phase 0b):
    {
        "id": "image_id",
        "conversations": [
            {"from": "user", "value": "Picture 1: <img>path</img>\nWhat disease..."},
            {"from": "assistant", "value": "Disease: Cassava Mosaic Disease (CMD)"}
        ],
        "metadata": {"class_name": "CMD", "label": 3, ...}
    }
    
    Format de sortie:
    {
        "conversations": [{"role": "user", ...}, {"role": "assistant", ...}],
        "image_path": "...",
        "class_label": "Cassava Mosaic Disease (CMD)",
        "class_idx": 3
    }
    """
    conversations = sample['conversations']
    metadata = sample.get('metadata', {})
    
    # Extraire image path
    image_path = None
    for turn in conversations:
        if '<img>' in turn['value']:
            image_path = extract_image_path(turn['value'])
            break
    
    # Obtenir la classe
    class_name = metadata.get('class_name', 'Unknown')
    label = metadata.get('label', 4)
    
    # Trouver le nom complet de la classe
    full_class_name = CLASS_NAMES[label] if 0 <= label < len(CLASS_NAMES) else 'Healthy'
    
    # Construire conversations pour Qwen
    formatted_conversations = [
        {
            "role": "user",
            "content": f"Picture 1: <image>\n{QUESTION}"
        },
        {
            "role": "assistant",
            "content": f"Disease: {full_class_name}"
        }
    ]
    
    return {
        "conversations": formatted_conversations,
        "image_path": image_path,
        "id": sample['id'],
        "class_label": full_class_name,
        "class_idx": label
    }

print("✅ Format functions defined")

In [ ]:
# ============================================================
# FORMATAGE DES DONNÉES
# ============================================================

print("\nFormatting data...")

train_formatted = [format_sample_for_qwen(s) for s in tqdm(cassava_train, desc="Train")]
val_formatted = [format_sample_for_qwen(s) for s in tqdm(cassava_val, desc="Val")]

# Mélanger
random.shuffle(train_formatted)

print(f"\n✅ Formatage terminé:")
print(f"  Train: {len(train_formatted):,}")
print(f"  Val:   {len(val_formatted):,}")

# Distribution
print("\n📊 Distribution train:")
train_dist = Counter(s['class_label'] for s in train_formatted)
for cls in CLASS_NAMES:
    count = train_dist.get(cls, 0)
    pct = count / len(train_formatted) * 100
    print(f"  {cls}: {count:,} ({pct:.1f}%)")

# Créer datasets
train_dataset = Dataset.from_list(train_formatted)
eval_dataset = Dataset.from_list(val_formatted)

In [ ]:
# Aperçu d'un sample
print("\n=== Aperçu Sample ===")
sample = train_formatted[0]
print(f"ID: {sample['id']}")
print(f"Class: {sample['class_label']}")
print(f"Image: .../{Path(sample['image_path']).name if sample['image_path'] else 'None'}")
print(f"\nUser: {sample['conversations'][0]['content'][:100]}...")
print(f"\nAssistant: {sample['conversations'][1]['content']}")

## 5. Data Collator

In [ ]:
# ============================================================
# AUGMENTATION (légère car données déjà augmentées en Phase 0b)
# ============================================================

try:
    import albumentations as A
    HAS_ALBUMENTATIONS = True
    
    # Augmentation minimale (données déjà pré-augmentées)
    train_transforms = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.3),
    ])
    print("✅ Albumentations - augmentation légère activée")
except ImportError:
    HAS_ALBUMENTATIONS = False
    train_transforms = None
    print("⚠️ Albumentations non disponible")

In [ ]:
# ============================================================
# DATA COLLATOR POUR QWEN
# ============================================================

@dataclass
class QwenVLDataCollator:
    """Data collator pour Qwen2.5-VL."""
    processor: Any
    transforms: Any = None
    is_training: bool = True

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        batch_messages = []
        batch_images = []
        batch_class_idx = []

        for feature in features:
            image_path = feature['image_path']
            image = None

            if image_path and os.path.exists(image_path):
                try:
                    image = Image.open(image_path).convert('RGB')
                    
                    # Augmentation
                    if self.is_training and self.transforms is not None:
                        image_np = np.array(image)
                        augmented = self.transforms(image=image_np)
                        image = Image.fromarray(augmented['image'])
                    
                    # Resize
                    image = image.resize(TARGET_IMAGE_SIZE, resample=Image.BILINEAR)
                except Exception as e:
                    # Image corrompue - utiliser placeholder
                    image = Image.new('RGB', TARGET_IMAGE_SIZE, color='white')
            else:
                image = Image.new('RGB', TARGET_IMAGE_SIZE, color='white')

            batch_images.append([image] if image else None)
            batch_class_idx.append(feature.get('class_idx', 4))

            # Construire messages pour Qwen
            conversations = feature['conversations']
            messages = []
            
            for turn in conversations:
                role = turn['role']
                content = turn['content']

                if role == 'user' and image is not None and len(messages) == 0:
                    # Premier message user avec image
                    messages.append({
                        "role": role,
                        "content": [
                            {"type": "image", "image": image},
                            {"type": "text", "text": content.replace('<image>', '').replace('Picture 1:', '').strip()}
                        ]
                    })
                else:
                    messages.append({
                        "role": role,
                        "content": content.replace('<image>', '').strip()
                    })

            batch_messages.append(messages)

        # Appliquer le chat template
        batch_texts_full = []
        batch_texts_user = []

        for messages in batch_messages:
            # Texte complet (user + assistant)
            full_text = self.processor.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False
            )
            batch_texts_full.append(full_text)

            # Texte user seulement (pour masquer dans labels)
            user_messages = [msg for msg in messages if msg["role"] == "user"]
            user_text = self.processor.tokenizer.apply_chat_template(
                user_messages, tokenize=False, add_generation_prompt=True
            )
            batch_texts_user.append(user_text)

        # Process avec le processor Qwen
        batch = self.processor(
            text=batch_texts_full,
            images=batch_images,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024
        )

        # Masquer les labels (user tokens = -100)
        labels = batch["input_ids"].clone()

        for i in range(len(batch_texts_user)):
            user_encoding = self.processor(
                text=[batch_texts_user[i]],
                images=[batch_images[i]],
                return_tensors="pt",
                padding=False,
                truncation=True,
                max_length=1024
            )
            user_length = user_encoding["input_ids"].shape[1]

            if user_length < labels[i].shape[0]:
                labels[i, :user_length] = -100
            else:
                labels[i, :-5] = -100

        batch["labels"] = labels
        batch["class_idx"] = torch.tensor(batch_class_idx)

        return batch

train_collator = QwenVLDataCollator(processor=processor, transforms=train_transforms, is_training=True)
eval_collator = QwenVLDataCollator(processor=processor, transforms=None, is_training=False)

print("✅ Data collators créés")

In [ ]:
# ============================================================
# TEST DATA COLLATOR
# ============================================================

print("\n" + "="*60)
print("TESTING DATA COLLATOR")
print("="*60)

test_features = [train_dataset[i] for i in range(2)]

try:
    test_batch = train_collator(test_features)
    print(f"✅ Batch créé avec succès!")
    print(f"   - input_ids shape: {test_batch['input_ids'].shape}")
    print(f"   - labels shape: {test_batch['labels'].shape}")
    
    for i in range(len(test_features)):
        active = (test_batch['labels'][i] != -100).sum().item()
        print(f"   - Sample {i}: {active} tokens actifs")
except Exception as e:
    print(f"❌ Erreur: {e}")
    import traceback
    traceback.print_exc()

## 6. Trainer et Entraînement

In [ ]:
# ============================================================
# TRAINER
# ============================================================

class CassavaTrainer(Trainer):
    """Trainer simplifié."""
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        inputs.pop('class_idx', None)
        outputs = model(**inputs)
        loss = outputs.loss if hasattr(outputs, 'loss') else outputs[0]
        return (loss, outputs) if return_outputs else loss

print("✅ CassavaTrainer configuré")

In [ ]:
# ============================================================
# CONFIGURATION TRAINING
# ============================================================

print("\n" + "="*60)
print("CONFIGURING TRAINER")
print("="*60)

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    
    # Batch
    per_device_train_batch_size=TRAINING_CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=TRAINING_CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=TRAINING_CONFIG["gradient_accumulation_steps"],
    
    # Training
    num_train_epochs=TRAINING_CONFIG["num_train_epochs"],
    learning_rate=TRAINING_CONFIG["learning_rate"],
    warmup_ratio=TRAINING_CONFIG["warmup_ratio"],
    lr_scheduler_type="cosine",
    
    # Regularization
    weight_decay=TRAINING_CONFIG["weight_decay"],
    max_grad_norm=TRAINING_CONFIG["max_grad_norm"],
    
    # Logging
    logging_steps=TRAINING_CONFIG["logging_steps"],
    save_steps=TRAINING_CONFIG["save_steps"],
    save_total_limit=3,
    
    # Evaluation
    eval_strategy="steps",
    eval_steps=TRAINING_CONFIG["eval_steps"],
    
    # Precision
    bf16=TRAINING_CONFIG["bf16"],
    fp16=TRAINING_CONFIG["fp16"],
    
    # Optimizer
    optim="adamw_8bit",
    
    # Best model
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    
    # Misc
    seed=RANDOM_SEED,
    report_to="none",
    remove_unused_columns=False,
    dataloader_pin_memory=False,
)

# Early stopping
early_stopping = EarlyStoppingCallback(
    early_stopping_patience=8,
    early_stopping_threshold=0.001
)

trainer = CassavaTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=train_collator,
    processing_class=tokenizer,
    callbacks=[early_stopping],
)

effective_batch = TRAINING_CONFIG['per_device_train_batch_size'] * TRAINING_CONFIG['gradient_accumulation_steps']
steps_per_epoch = len(train_dataset) // effective_batch
total_steps = steps_per_epoch * TRAINING_CONFIG['num_train_epochs']

print(f"\nConfiguration:")
print(f"  - Train samples: {len(train_dataset):,}")
print(f"  - Effective batch: {effective_batch}")
print(f"  - Steps per epoch: ~{steps_per_epoch:,}")
print(f"  - Total steps: ~{total_steps:,}")
print(f"  - LR: {TRAINING_CONFIG['learning_rate']}")
print(f"  - Epochs: {TRAINING_CONFIG['num_train_epochs']}")
print("="*60)

In [ ]:
# ============================================================
# ENTRAÎNEMENT
# ============================================================

print("\n" + "="*60)
print("STARTING TRAINING - STAGE 1 KAGGLE-ONLY")
print("="*60)
print(f"\n🎯 Objectifs:")
print(f"  - Test accuracy ≥70% (minimum pour Phase 1)")
print(f"  - Test accuracy ≥75% (idéal)")
print(f"  - Toutes classes ≥60% F1")
print(f"\n⏱️ Temps estimé: ~8-10 heures")
print("="*60 + "\n")

train_result = trainer.train()

print("\n" + "="*60)
print("TRAINING COMPLETED")
print("="*60)
print(f"Final loss: {train_result.training_loss:.4f}")
print(f"Total steps: {train_result.global_step}")

## 7. Sauvegarde du Modèle

In [ ]:
# ============================================================
# SAUVEGARDE
# ============================================================

print("\nSaving model...")
final_model_path = CHECKPOINT_DIR / "final_model"
model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)

# Sauvegarder config
config = {
    "phase": "0c",
    "description": "Stage 1 trained on 100% Kaggle data",
    "model_name": MODEL_NAME,
    "lora_config": LORA_CONFIG,
    "training_config": TRAINING_CONFIG,
    "training_loss": train_result.training_loss,
    "total_steps": train_result.global_step,
    "train_samples": len(train_dataset),
    "val_samples": len(eval_dataset),
}

with open(CHECKPOINT_DIR / "training_config.json", 'w') as f:
    json.dump(config, f, indent=2)

print(f"\n✅ Modèle sauvegardé: {final_model_path}")
print(f"✅ Config sauvegardée: {CHECKPOINT_DIR / 'training_config.json'}")

## 8. Évaluation Rapide

In [ ]:
# ============================================================
# ÉVALUATION RAPIDE SUR VALIDATION
# ============================================================

print("\n" + "="*60)
print("QUICK EVALUATION ON VALIDATION SET")
print("="*60)

# Mode inference
FastVisionModel.for_inference(model)

def extract_prediction(response: str) -> str:
    """Extrait la prédiction de la réponse."""
    response_lower = response.lower()
    
    # Chercher "Disease: XXX"
    if "disease:" in response_lower:
        match = re.search(r'disease:\s*([\w\s()]+)', response_lower)
        if match:
            disease_text = match.group(1).strip()
            
            # Mapper vers classe
            if "cbb" in disease_text or "bacterial blight" in disease_text:
                return "Cassava Bacterial Blight (CBB)"
            elif "cbsd" in disease_text or "brown streak" in disease_text:
                return "Cassava Brown Streak Disease (CBSD)"
            elif "cgm" in disease_text or "green mottle" in disease_text:
                return "Cassava Green Mottle (CGM)"
            elif "cmd" in disease_text or "mosaic" in disease_text:
                return "Cassava Mosaic Disease (CMD)"
            elif "healthy" in disease_text:
                return "Healthy"
    
    return "Unknown"

def predict_single(sample, max_new_tokens=50):
    """Prédit pour un sample."""
    image_path = sample['image_path']
    
    try:
        image = Image.open(image_path).convert('RGB').resize(TARGET_IMAGE_SIZE)
    except:
        return "Error", ""
    
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": QUESTION}
            ]
        }
    ]
    
    input_text = processor.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    inputs = processor(
        text=[input_text],
        images=[[image]],
        return_tensors="pt",
        padding=True,
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    
    response = processor.tokenizer.decode(outputs[0], skip_special_tokens=True)
    predicted_class = extract_prediction(response)
    
    return predicted_class, response

# Test rapide sur 100 samples validation
print("\n--- Quick Test (100 samples) ---")
quick_samples = random.sample(val_formatted, min(100, len(val_formatted)))
quick_results = []

for sample in tqdm(quick_samples, desc="Quick test"):
    pred, raw = predict_single(sample)
    quick_results.append({
        'gt': sample['class_label'],
        'pred': pred,
        'correct': pred == sample['class_label']
    })

correct = sum(1 for r in quick_results if r['correct'])
print(f"\n📊 Quick Val Accuracy: {correct}%")

print("\n📈 Par classe:")
for cls in CLASS_NAMES:
    cls_samples = [r for r in quick_results if r['gt'] == cls]
    if cls_samples:
        cls_correct = sum(1 for r in cls_samples if r['correct'])
        acc = cls_correct / len(cls_samples) * 100
        status = "✅" if acc >= 60 else "⚠️" if acc >= 40 else "❌"
        print(f"  {status} {cls}: {acc:.1f}% ({cls_correct}/{len(cls_samples)})")

## 9. Résumé et Prochaines Étapes

In [ ]:
print("="*60)
print("PHASE 0c TERMINÉE: Stage 1 Kaggle-Only Entraîné")
print("="*60)
print()
print("📊 RÉSUMÉ")
print(f"  - Modèle: {MODEL_NAME}")
print(f"  - Train samples: {len(train_dataset):,}")
print(f"  - Val samples: {len(eval_dataset):,}")
print(f"  - Training loss: {train_result.training_loss:.4f}")
print(f"  - Total steps: {train_result.global_step}")
print()
print("📂 CHECKPOINT")
print(f"  {CHECKPOINT_DIR / 'final_model'}")
print()
print("✅ PROCHAINE ÉTAPE")
print("  Phase 0d: Évaluer sur le test set Kaggle (3,774 images)")
print("  → Utiliser notebook: 12_evaluate_stage1_on_test.ipynb")
print("  → Changer STAGE1_CHECKPOINT vers 'stage1_kaggle_only/final_model'")
print()
print("🎯 OBJECTIF PHASE 0d")
print("  - Test accuracy ≥70% → Procéder à Phase 1 (SMoLoRA)")
print("  - Test accuracy <70% → Itérer sur Stage 1")
print()
print("="*60)